In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, col, explode, lower, regexp_extract

## Initiate Spark session

In [2]:
from pyspark.sql import SparkSession

# Initialize a Spark session
spark = (SparkSession
         .builder
         .appName("Analyzing the vocabulary of Pride and Prejudice")
         .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/04 16:02:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [14]:
# Access the Spark context from the Spark session
spark.sparkContext

<SparkContext master=local[*] appName=pyspark-shell>

In [15]:
# Set the log level to ERROR to reduce verbosity
spark.sparkContext.setLogLevel("ERROR")

In [16]:
# Create a simple DataFrame
data = [
        ("Alice", 34, "NY"),
        ("Bob", 45, "CA"),
        ("Cathy", 29, "WA"),
    ]
columns = ["name", "age", "state"]
df = spark.createDataFrame(data, schema=columns)
df.show(truncate=False)

+-----+---+-----+
|name |age|state|
+-----+---+-----+
|Alice|34 |NY   |
|Bob  |45 |CA   |
|Cathy|29 |WA   |
+-----+---+-----+



In [17]:
# List the methods and attributes of the DataFrame
dir(df)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_collect_as_arrow',
 '_ipython_key_completions_',
 '_jcols',
 '_jdf',
 '_jmap',
 '_joinAsOf',
 '_jseq',
 '_lazy_rdd',
 '_repr_html_',
 '_sc',
 '_schema',
 '_session',
 '_show_string',
 '_sort_cols',
 '_sql_ctx',
 '_support_repr_html',
 'age',
 'agg',
 'alias',
 'approxQuantile',
 'cache',
 'checkpoint',
 'coalesce',
 'colRegex',
 'collect',
 'columns',
 'corr',
 'count',
 'cov',
 'createGlobalTempView',
 'createOrReplaceGlobalTempView',
 'createOrReplaceTempView',
 'createTempView',
 'crossJoin',
 'crosstab',
 'cube',
 'describe',
 'distinct',
 'drop',
 'dropDuplicates',
 'dropDuplicatesWi

In [18]:
# Print the documentation for the printSchema function
print(df.printSchema.__doc__)

Prints out the schema in the tree format.
        Optionally allows to specify how many levels to print if schema is nested.

        .. versionadded:: 1.3.0

        .. versionchanged:: 3.4.0
            Supports Spark Connect.

        Parameters
        ----------
        level : int, optional, default None
            How many levels to print for nested schemas.

            .. versionchanged:: 3.5.0
                Added Level parameter.

        Examples
        --------
        >>> df = spark.createDataFrame(
        ...     [(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])
        >>> df.printSchema()
        root
         |-- age: long (nullable = true)
         |-- name: string (nullable = true)

        >>> df = spark.createDataFrame([(1, (2,2))], ["a", "b"])
        >>> df.printSchema(1)
        root
         |-- a: long (nullable = true)
         |-- b: struct (nullable = true)

        >>> df.printSchema(2)
        root
         |-- a: long (nullable = true)
   

## Read data

In [3]:
# Read the text file into a DataFrame
book = spark.read.text("../../data/gutenberg_books/1342-0.txt")

# Display the schema of the DataFrame
book

DataFrame[value: string]

In [4]:
# Display the schema of the DataFrame
book.printSchema()

root
 |-- value: string (nullable = true)



In [21]:
# Show the first 5 rows of the DataFrame with a truncate of 50 characters
book.show(5, truncate=50)

+--------------------------------------------------+
|                                             value|
+--------------------------------------------------+
|The Project Gutenberg EBook of Pride and Prejud...|
|                                                  |
|This eBook is for the use of anyone anywhere at...|
|almost no restrictions whatsoever.  You may cop...|
|re-use it under the terms of the Project Gutenb...|
+--------------------------------------------------+
only showing top 5 rows



## Tokenize each word

In [5]:
# Split each line into words and create a new DataFrame
lines = book.select(split(col("value"), " ").alias("line"))
lines.show(5, truncate=False)

+----------------------------------------------------------------------------------+
|line                                                                              |
+----------------------------------------------------------------------------------+
|[The, Project, Gutenberg, EBook, of, Pride, and, Prejudice,, by, Jane, Austen]    |
|[]                                                                                |
|[This, eBook, is, for, the, use, of, anyone, anywhere, at, no, cost, and, with]   |
|[almost, no, restrictions, whatsoever., , You, may, copy, it,, give, it, away, or]|
|[re-use, it, under, the, terms, of, the, Project, Gutenberg, License, included]   |
+----------------------------------------------------------------------------------+
only showing top 5 rows



In [ ]:
# Explode the lines into individual words
words = lines.select(explode(col("line")).alias("word"))
words.show(15, truncate=False)

+----------+
|word      |
+----------+
|The       |
|Project   |
|Gutenberg |
|EBook     |
|of        |
|Pride     |
|and       |
|Prejudice,|
|by        |
|Jane      |
|Austen    |
|          |
|This      |
|eBook     |
|is        |
+----------+
only showing top 15 rows



## Clean Data

In [ ]:
# Convert all words to lowercase
words_lower = words.select(lower(col("word")).alias("word_lower"))
words_lower.show(15, truncate=False)

+----------+
|word_lower|
+----------+
|the       |
|project   |
|gutenberg |
|ebook     |
|of        |
|pride     |
|and       |
|prejudice,|
|by        |
|jane      |
|austen    |
|          |
|this      |
|ebook     |
|is        |
+----------+
only showing top 15 rows



In [ ]:
# Remove punctuation and non-alphabetic characters
words_clean = words_lower.select(regexp_extract(col("word_lower"), "[a-z]+", 0).alias("word_clean"))
words_clean.show(15, truncate=False)

+----------+
|word_clean|
+----------+
|the       |
|project   |
|gutenberg |
|ebook     |
|of        |
|pride     |
|and       |
|prejudice |
|by        |
|jane      |
|austen    |
|          |
|this      |
|ebook     |
|is        |
+----------+
only showing top 15 rows



In [ ]:
# Filter out empty strings
words_nonull = words_clean.filter(col("word_clean") != "")
words_nonull.show(15, truncate=False)

+----------+
|word_clean|
+----------+
|the       |
|project   |
|gutenberg |
|ebook     |
|of        |
|pride     |
|and       |
|prejudice |
|by        |
|jane      |
|austen    |
|this      |
|ebook     |
|is        |
|for       |
+----------+
only showing top 15 rows



## Stop Spark session

In [ ]:
# Stop the Spark session
spark.stop()